In [1]:
!pip install kagglehub

In [2]:
import kagglehub
import os

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Dataset downloaded to:", path)


Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Dataset downloaded to: /kaggle/input/chest-xray-pneumonia


In [3]:
DATASET_DIR = os.path.join(path, "chest_xray")

In [4]:
print(os.listdir(DATASET_DIR))
print(os.listdir(os.path.join(DATASET_DIR, "train")))


['chest_xray', '__MACOSX', 'val', 'test', 'train']
['PNEUMONIA', 'NORMAL']


In [ ]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam


from sklearn.metrics import classification_report


IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10


OUTPUT_DIR = "Pneumonia_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
print("Train classes:", os.listdir(os.path.join(DATASET_DIR, "train")))
print("Test classes:", os.listdir(os.path.join(DATASET_DIR, "test")))

Train classes: ['PNEUMONIA', 'NORMAL']
Test classes: ['PNEUMONIA', 'NORMAL']


In [7]:
train_gen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)


train_data = train_gen.flow_from_directory(
os.path.join(DATASET_DIR, "train"),
target_size=(IMG_SIZE, IMG_SIZE),
batch_size=BATCH_SIZE,
class_mode='binary'
)


val_data = val_gen.flow_from_directory(
os.path.join(DATASET_DIR, "val"),
target_size=(IMG_SIZE, IMG_SIZE),
batch_size=BATCH_SIZE,
class_mode='binary'
)


test_data = test_gen.flow_from_directory(
os.path.join(DATASET_DIR, "test"),
target_size=(IMG_SIZE, IMG_SIZE),
batch_size=1,
shuffle=False,
class_mode='binary'
)


class_mapping = {v: k for k, v in train_data.class_indices.items()}
print("Class mapping:", class_mapping)

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Class mapping: {0: 'NORMAL', 1: 'PNEUMONIA'}


In [8]:
model = Sequential([
Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
MaxPooling2D(2,2),


Conv2D(64, (3,3), activation='relu'),
MaxPooling2D(2,2),


Conv2D(128, (3,3), activation='relu'),
MaxPooling2D(2,2),


Flatten(),
Dense(128, activation='relu'),
Dropout(0.5),
Dense(1, activation='sigmoid')
])


model.compile(
optimizer=Adam(learning_rate=1e-4),
loss='binary_crossentropy',
metrics=['accuracy']
)


model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
train_data,
validation_data=val_data,
epochs=EPOCHS
)


model.save(os.path.join(OUTPUT_DIR, "cnn_pneumonia_model.h5"))

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 93s 521ms/step - accuracy: 0.8054 - loss: 0.4489 - val_accuracy: 0.7500 - val_loss: 0.5683
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 55s 338ms/step - accuracy: 0.9448 - loss: 0.1416 - val_accuracy: 0.6875 - val_loss: 0.7684
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 54s 332ms/step - accuracy: 0.9560 - loss: 0.1179 - val_accuracy: 0.8125 - val_loss: 0.3419
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 55s 335ms/step - accuracy: 0.9606 - loss: 0.1041 - val_accuracy: 0.6875 - val_loss: 0.6522
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 54s 332ms/step - accuracy: 0.9673 - loss: 0.0897 - val_accuracy: 0.9375 - val_loss: 0.1902
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 55s 339ms/step - accuracy: 0.9669 - loss: 0.0888 - val_accuracy: 0.6875 - val_loss: 0.5594
Epoch 7/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 55s 335ms/step - accuracy: 0.9768 - loss: 0.0690 - val_accuracy: 0.8125 - val_loss: 0.3179
Epoch 8/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 54s 332ms/step - accuracy: 0.9735 - loss: 0

In [10]:
preds = model.predict(test_data)
pred_labels = (preds > 0.5).astype(int).ravel()
true_labels = test_data.classes

print(classification_report(true_labels, pred_labels, target_names=list(class_mapping.values())))

624/624 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step
              precision    recall  f1-score   support

      NORMAL       0.98      0.37      0.54       234
   PNEUMONIA       0.73      0.99      0.84       390

    accuracy                           0.76       624
   macro avg       0.85      0.68      0.69       624
weighted avg       0.82      0.76      0.73       624



In [ ]:
pneumonia_results = []

for i, file_path in enumerate(test_data.filepaths):
    pneumonia_results.append({
        "image_id": os.path.basename(file_path),
        "prediction": class_mapping[pred_labels[i]],
        "confidence": float(preds[i][0]),
        "agent_task": "medical_image_analysis",
        "model": "CNN_Pneumonia_v1"
    })

# Sauvegarde en JSON
with open(os.path.join(OUTPUT_DIR, "pneumonia_predictions.json"), "w") as f:
    json.dump(pneumonia_results, f, indent=4)

# Sauvegarde en CSV
pd.DataFrame(pneumonia_results).to_csv(
    os.path.join(OUTPUT_DIR, "pneumonia_predictions.csv"),
    index=False
)

print("\nPneumonia outputs generated successfully:")
print("- pneumonia_output/pneumonia_predictions.json")
print("- pneumonia_output/pneumonia_predictions.csv")


JADE outputs generated successfully:
- jade_output/jade_predictions.json
- jade_output/jade_predictions.csv
